In [ ]:
with cohorts as (

    select
        client_id,
        campaigns_cnt,
        first_campaign_dt::date as first_campaign_dt,
        date_trunc('month', first_campaign_dt)::date as first_campaign_month

    from cvm_sbx.{prefix}_CVMB_24118_client_cohorts

    where first_campaign_dt is not null

),

cheque_filtered as (

    select
        contact_id,
        datetime,
        summ_discounted

    from dm.cheque

    where operation_type_id = 1
      and summ_discounted > 0

      -- общий минимально нужный период
      and datetime >= date '2025-11-01'
      and datetime < date '2026-04-30'

)

select
    c.client_id,
    c.campaigns_cnt,
    c.first_campaign_dt,

    date_trunc('month', ch.datetime)::date as month_dt,

    sum(ch.summ_discounted) as month_spend

from cohorts c

join cheque_filtered ch
    on ch.contact_id = c.client_id
    and ch.datetime >= c.first_campaign_month - interval '1 month'
    and ch.datetime < date '2026-04-30'

group by
    c.client_id,
    c.campaigns_cnt,
    c.first_campaign_dt,
    month_dt

order by
    c.client_id,
    month_dt;

In [ ]:
import pandas as pd


df['first_campaign_dt'] = pd.to_datetime(df['first_campaign_dt'])
df['month_dt'] = pd.to_datetime(df['month_dt'])

df['month_shift'] = (
    (df['month_dt'].dt.year - df['first_campaign_dt'].dt.year) * 12
    + (df['month_dt'].dt.month - df['first_campaign_dt'].dt.month)
)

wide_df = (
    df
    .pivot_table(
        index=[
            'client_id',
            'campaigns_cnt',
            'first_campaign_dt'
        ],
        columns='month_shift',
        values='month_spend',
        aggfunc='sum',
        fill_value=0
    )
    .reset_index()
)

wide_df.columns = [
    f'spend_m{abs(col)}' if isinstance(col, int) and col < 0
    else f'spend_p{col}' if isinstance(col, int) and col > 0
    else 'spend_0' if col == 0
    else col
    for col in wide_df.columns
]

wide_df.head()